# Quick Overview: Cross-Correlation and Likelihood Mapping


This section performs cross-correlation analysis to detect planetary signals, following methods outlined in 
Boucher et al. (2021, 2023) and Gibson et al. (2020). Two approaches are provided:

Injection Correlation:
   - Injects a model at each Kp across all exposures.
   - More computationally intensive, but better for modeling and recovering full transmission spectra.

Functions also include likelihood calculation, velocity models, and plotting utilities for correlation maps and significance testing. Our tutorial will only run through the injected correlation method.

<span style="font-size:24px; font-weight:bold; color:white; background-color:blue; display:block; padding:10px;">
Welcome to the workshop version of this notebook! This notebook will NOT fully run through. We have placed highlighted questions to help guide you with what needs to be fixed/changed. If needed feel free to take a peek at the answer key if you are stuck!
</span>

<details>
  <summary style="font-weight:bold; color:green; cursor:pointer;">✅ Click here to show the answer</summary>

  all the answers/ hints will be shown in these drop down menus!
</details>

In [ ]:
#we are NOT running petitRADTRANS in this tutorial however, starships needs it to run. 
#we are tricking the container here to believe we have petitRADTRANS when infact all the functions are empty

import sys
import types

# Top-level dummy package
petitRADTRANS = types.ModuleType("petitRADTRANS")

# Submodules as separate mock modules/namespaces
petitRADTRANS.Radtrans = object  # or define a dummy class

petitRADTRANS.nat_cst = types.SimpleNamespace()
petitRADTRANS.physics = types.SimpleNamespace(
    guillot_global=lambda *a, **k: None,
    guillot_modif=lambda *a, **k: None
)
petitRADTRANS._read_opacities = types.SimpleNamespace()
petitRADTRANS.fort_input = types.SimpleNamespace()
petitRADTRANS.fort_rebin = types.SimpleNamespace()
petitRADTRANS.pyth_input = types.SimpleNamespace()

poor_mans_nonequ_chem = types.ModuleType("poor_mans_nonequ_chem")
poor_mans_nonequ_chem.interpol_abundances = lambda *a, **k: None

# Register everything in sys.modules
sys.modules["petitRADTRANS"] = petitRADTRANS
sys.modules["petitRADTRANS.radtrans"] = petitRADTRANS
sys.modules["petitRADTRANS._read_opacities"] = petitRADTRANS._read_opacities
sys.modules["petitRADTRANS.fort_input"] = petitRADTRANS.fort_input
sys.modules["petitRADTRANS.fort_rebin"] = petitRADTRANS.fort_rebin
sys.modules["petitRADTRANS.pyth_input"] = petitRADTRANS.pyth_input
sys.modules["petitRADTRANS.nat_cst"] = petitRADTRANS.nat_cst
sys.modules["petitRADTRANS.physics"] = petitRADTRANS.physics
sys.modules["petitRADTRANS.poor_mans_nonequ_chem"] = poor_mans_nonequ_chem

In [ ]:
# === Set up plotting inline for Jupyter ===
%matplotlib inline

# === Standard Library Imports ===
import os
import logging
import warnings
from pathlib import Path
from sys import path
from itertools import product

# === Scientific Libraries ===
import numpy as np
import matplotlib.pyplot as plt
import astropy.units as u
import astropy.constants as const

# === Starships Module Imports ===
import starships.plotting_fcts as pf
from starships import homemade as hm
import starships.correlation_class as cc
from starships.correlation_class import Correlations
import starships.correlation as corr
import starships.planet_obs as pl_obs
from starships.planet_obs import Observations, Planet

# === Miscellaneous Settings ===

# Suppress common warnings for cleaner output
warnings.simplefilter("ignore", UserWarning)
warnings.simplefilter("ignore", RuntimeWarning)

# Suppress verbose fontTools logging output
logging.getLogger("fontTools.subset").setLevel(logging.WARNING)

# === Load custom colors for plotting ===
couleurs = hm.get_colors('magma', 50)[5:-2]


# Load in the reduction files

This section loads the reduction files that were generated in the previous notebook.
We'll also define the planet parameters here — make sure they match the ones used in the earlier reduction step.

<span style="font-size:24px; font-weight:bold; color:white; background-color:blue; display:block; padding:10px;">
Some of these parameters are wrong! Make sure they match your reduction files!
</span>

<details>
  <summary style="font-weight:bold; color:green; cursor:pointer;">✅ Click here to show the answer</summary>

  M_star, Period and inclination are wrong
</details>

In [ ]:

#input planet name this will pull from exofile, so can change any parameters here
# === Planetary and Stellar Parameters that can be changed===
#make sure to set the same as the reduction notebook

pl_name = 'WASP-127 b'
planet_obj = Planet(pl_name)

# Stellar parameters
planet_obj.M_star = 1.950 * const.M_sun     # Mass of the star
planet_obj.R_star = 1.333 * u.R_sun         # Radius of the star
planet_obj.Teff = 5842 * u.K                # Effective temperature of the star

# Planetary parameters
planet_obj.M_pl = 0.165 * u.M_jup           # Mass of the planet
planet_obj.R_pl = 1.311 * u.R_jup           # Radius of the planet
planet_obj.Tp = 1400 * u.K                  # Planet temperature

# Orbital parameters
planet_obj.period = 6.178062 * u.day        # Orbital period
planet_obj.trandur = 0.181 * u.day          # Transit duration
planet_obj.ap = 0.04840 * u.au              # Semi-major axis
planet_obj.incl = 85.85 * u.deg             # Orbital inclination
planet_obj.excent = 0.0                     # Eccentricity
planet_obj.w = (-90 * u.deg).to(u.rad)      # Argument of periastron (converted to radians)

In [ ]:
#give a path to the files we reduced in the previous notebook, this is the same as your outdir in the previous notebook
reduc_dir = '/home/jovyan/Notebooks/WASP-127b_ExoSLAM_Workshop'


In [ ]:
#list the file names you would like to use here
filename_list = ['sequence_2-pc_mask_wings90_data_trs_WASP127b_TR1']
obs_list = []

#this will run through all of the name and create an "obs_list"(a list of our observations)

for filename in filename_list:
    obs = pl_obs.load_single_sequences(filename, pl_name, path=reduc_dir,
                              load_all=False, filename_end='', plot=False, planet=planet_obj)
    obs_list.append(obs)
    
#the obs list will have all of the information from the reduction files

# Next we Need to load in the Model
In this notebook, we aren't supposed to build the model from scratch. Instead, the model should have already been created in a previous notebook, often called something like make_a_model. That earlier notebook uses PetitRADTRANS (PRT) to generate a planetary atmosphere model and saves it to a file.

Here, we simply load that precomputed model so we can analyze it or compare it to data. This approach saves time and ensures consistent results, because generating the model with PRT can take a while and depends on many input parameters.

PetitRADTRANS is a tool for simulating the transmission or emission spectra of exoplanet atmospheres, based on properties like temperature, composition, and pressure.

In [ ]:
# Path to and name of the model file
model_file = np.load('/home/jovyan/Models/model_NIRPS-APERO_H2O_main_iso_FeH_main_iso_CO_all_iso.npz')
wave_mod_comb, model_spec_comb = model_file['wave_mod'], model_file['mod_spec']

# Plot it
plt.plot(wave_mod_comb, model_spec_comb)
plt.title('model_NIRPS-APERO_H2O_main_iso_FeH_main_iso_CO_all_is')

<span style="font-size:24px; font-weight:bold; color:white; background-color:blue; display:block; padding:10px;">
Can you check out the other models the same was as above? Can you overplot them? What is the difference between them
</span>

<details>
  <summary style="font-weight:bold; color:green; cursor:pointer;">✅ Click here to show the answer</summary>

  There is another model called "model_NIRPS-APERO_H2O_pokazatel_main_iso.npz". 
</details>

In [ ]:
model_file_h20 = np.load('/home/jovyan/Models/  ')
wave_mod_h20, model_spec_h20 = model_file_h20[' '], model_file[' ']

plt.plot(wave_mod, )
plt.title(' ')

# Lets run the Injection Correlation

First we need to define a few extra parameters and then we are ready to run!

In [ ]:
#where do you want the proucts stored?
output_dir = '/home/jovyan/Notebooks/WASP-127b_ExoSLAM_Workshop/'

print(output_dir)

In [ ]:
corr.calc_logl_injred?

<span style="font-size:24px; font-weight:bold; color:white; background-color:blue; display:block; padding:10px;">
Fill in some of the missing parameters such as kind_trans, and visit name
</span>

<details>
  <summary style="font-weight:bold; color:green; cursor:pointer;">✅ Click here to show the answer</summary>

  Kind trans is 'transmission' and visit_name should be similar to that in the reduction so something like "WASP127b_TR1"
</details>

In [ ]:
# === Define parameters for correlation ===

#define the visit name for file naming
visit_name = 'Aug012023'

# Define model type and type of data (transmission or emission)
kind_trans = 'transmission'

# Define RV grid for injected signal
n_RV_inj = 31
corrRV0 = np.linspace(-30, 30, n_RV_inj)  # RVs for cross-correlation


# ===== Set up the model to cross correlate with ======

#define the model you wish to use to cross corelated
wave_mod, model_spec = model_file_h20['wave_mod'], model_file_h20['mod_spec']

#model name that you are using 
model_name = 'H2O'


#file name for the reduction you would like to cross correlate [can be a list]
filename_list = [f'sequence_{n_pc}-pc_mask_wings{mask_w}_data_trs_WASP127b_TR1'
                 for mask_w, n_pc in product([90], range(2, 3))]
print('\n'.join(filename_list))


# # === Initialize storage for results ===


n_pc_list = []
mask_wings_list = []
all_obs = dict()
all_ccf_map = dict()
all_logl_map = dict()


# # === loop through filelist to cross correlate if desired ===

for filename in filename_list:
    obs = pl_obs.load_single_sequences(filename, pl_name, path=reduc_dir,
                              load_all=False, filename_end='', plot=False, planet=planet_obj)
    
    # Generate Kp from the reduction file
    Kp_array = np.array([obs.Kp.value]) 
        
    n_pc = int(obs.params[5])
    n_pc_list.append(n_pc)
    print('n_pc_list',n_pc_list)
    
    mask_wings = int(obs.params[1] * 100)  # in percent
    mask_wings_list.append(mask_wings)
    print('mask_wings_list',mask_wings_list)

    out_filename = f'{Path(filename).stem}_ccf_logl_seq_{model_name}'

    ccf_map, logl_map = corr.calc_logl_injred(
        obs,'seq', planet_obj, Kp_array, corrRV0, [n_pc], wave_mod, model_spec,  kind_trans
    )


    corr.save_logl_seq(output_dir / Path(out_filename), ccf_map, logl_map,
                        wave_mod, model_spec, n_pc, Kp_array, corrRV0, kind_trans)


    all_obs[(n_pc, mask_wings)] = obs
    all_ccf_map[(n_pc, mask_wings)] = ccf_map
    all_logl_map[(n_pc, mask_wings)] = logl_map

# Thats it! Lets look at the results!

This can also be done in a seperate notebook

In [ ]:
cc.plot_ccflogl?

<span style="font-size:24px; font-weight:bold; color:white; background-color:blue; display:block; padding:10px;">
What happens to the detection if you changes the orders we are looking at?
</span>

<details>
  <summary style="font-weight:bold; color:green; cursor:pointer;">✅ Click here to show the answer</summary>

  The detection might change based on the orders we use! for water since it stretches the whole model we will want to use all the order but form something like CO that is only in a small range, we only need specific orders
</details>

In [ ]:
# === Plot Single CCF and Log-Likelihood for a Given PCA and Masking Setup ===

# Select the PCA number and mask wings percentage to analyze
n_pc, mask_w = 2, 90

# Define the spectral orders to include in the cross-correlation function (CCF)
# You can specify a single order as an integer or multiple orders as a list/array
order_indices = np.array([60])  # Example: order 60 only
# order_indices = [46, 47]      # Example for multiple orders (uncomment to use)

print(f"Orders used for CCF and log-likelihood calculation: {order_indices}")

# Define the filename for the reduction file to analyze
filename = f'sequence_{n_pc}-pc_mask_wings{mask_w}_data_trs_WASP127b_TR1'

# Load the observation data for the specified filename and planet
obs = pl_obs.load_single_sequences(
    filename, pl_name, path=reduc_dir,
    load_all=False, filename_end='', plot=False,
    planet=planet_obj
)

# Retrieve the relevant precomputed objects from dictionaries by (n_pc, mask_wings) key from the above cell
args = [all_something[(n_pc, mask_wings)] for all_something in [all_obs, all_ccf_map, all_logl_map]]

# Compute and plot the CCF and log-likelihood objects for the specified orders
# 'plot_ccflogl' returns ccf and logl objects, and optionally plots figures
ccf_obj, logl_obj = cc.plot_ccflogl(
    *args, #pulling in the save directories from above
    corrRV0, Kp_array, [n_pc], #defined above
    RV=-8.9,
    orders=order_indices, #defined above
    path_fig=" ",    # Path to save figures (empty means no save)
    fig_name=" "     # Figure name prefix (empty means default)
)



# Generate the T-test

The t-test will be used to evaluate the null hypothesis that the two samples—typically in-transit and out-of-transit data—are drawn from the same underlying distribution. In this context, the test assesses whether any observed difference between the two datasets (such as the strength of a spectral feature or cross-correlation signal) is statistically significant or simply due to random noise. If the graph of in and out look the same, there is no clear detection.

In [ ]:
# === Generate the t-test map using the CCF object ===

# Parameters can be adjusted here to refine the map calculation
ccf_obj.ttest_map(
    all_obs[(n_pc, mask_wings)],      # Observation object for selected PCA and mask wings
    kind='logl',                      # Use log-likelihood method for t-test
    vrp=np.zeros_like(obs.vrp),       # Velocity residual profile (set to zero array here)
    orders=order_indices,             # Spectral orders to include in the analysis
    kp0=0,                           # Initial Kp value to start search (km/s)
    RV_limit=25,                    # Radial velocity limit (km/s)
    kp_step=5,                      # Step size for Kp grid (km/s)
    rv_step=2,                      # Step size for RV grid (km/s)
    RV=None,                        # Optional RV shift parameter
    speed_limit=3,                  # Speed limit (km/s) for smoothing or constraints
    equal_var=False                 # Assume unequal variance in t-test
)


# Normalize the significance with three different methodes!

There are multiple ways to normalize and properly quanitfy the significance. We can use a Ttest with a full scale normalization, a box methode or a sigma clipping method. We show all three below to show a comparison in quantifying the detection.

1) To accurately represent the significance in a Kp/Vsys map, we normalize the t_value map so that the minimum corresponds to -3σ. This ensures consistency with our sigma claims. We can use only certain part of the test map, or scale the entire map.

2) the "box" method, where we just divide the median-subtracted SNR map by the standard deviation of the values in a pre-defined box far away from the signal (which I think is the most common thing people do these days)

3) the sigma-clipping method, which has been used more recently by Kasper+ 2021, Mansfield+ 2024 and others.

# Notes need to add more about sigma clipping -> Georgia is unsure how to explain it well



<span style="font-size:24px; font-weight:bold; color:white; background-color:blue; display:block; padding:10px;">
Try to all three normalizations. Does anything change?
</span>

<details>
  <summary style="font-weight:bold; color:green; cursor:pointer;">✅ Click here to show the answer</summary>

  Yes, the normalization will change the detection significance! 
</details>

In [ ]:
# === Retrieve Objects for the Selected PCA and Mask Wings ===

# Pull the observation object for the selected (n_pc, mask_wings) key
tr = all_obs[(n_pc, mask_wings)]

# Use the previously computed cross-correlation function (CCF) object
cobj = ccf_obj

# Extract the t-test statistic map from the CCF object
t_value = cobj.ttest_map_tval


# === Scale the t-test map values ===

# Scale the entire t_value map so that its minimum maps to -3 (arbitrary scaling)
t_value_scaled = t_value * (-3) / t_value.min()

# Example: scale only part of the map (uncomment and adjust as needed)
# kp_index = cobj.ttest_map_kp < 140
# t_value_scaled = t_value * (-3) / np.nanmin(t_value[kp_index, :])


# === Plot the t-test map histogram with provided parameters ===

(t_in, p_in) = pf.plot_ttest_map_hist(
    tr,                     # Observation object
    cobj.rv_grid,           # Radial velocity grid
    cobj.map_prf.copy(),    # CCF map (profile)
    cobj.ttest_map_kp,      # Kp values from t-test map
    cobj.ttest_map_rv,      # RV values from t-test map
    t_value_scaled,         # Scaled t-test statistic map
    cobj.ttest_map_params,  # Additional t-test parameters
    plot_trail=True,        # Whether to plot trail
    masked=True,            # Apply masking to data
    ccf=cobj.map_prf.copy(),# Provide CCF profile again
    vrp=np.zeros_like(tr.vrp),  # Zero velocity residual profile for plotting
    RV=cobj.pos,            # RV position of detection
    hist=False,             # Disable histogram plot inside function
    show_max=False,         # Do not highlight max by default
    show_rest_frame=False,  # Do not shift to rest frame
    fig_name='',            # Figure name (empty = no save)
    path_fig=None,          # Path to save figure (None = no save)
    orders=order_indices    # Spectral orders to include
)




In [ ]:
def calculate_KpVsys_map(OBS_OBJECT, CCF_OBJECT, Kp_list, vsys_list, method='box'):

    import matplotlib.gridspec as gridspec
    import matplotlib
    from astropy import stats

    ######################################
    # 1. Build coordinate grid for plotting
    ######################################
    # Extend the coordinate arrays by one step for plotting with pcolormesh
    x_coords = np.concatenate((vsys_list, [vsys_list[-1]+np.diff(vsys_list)[-1]]))
    y_coords = np.concatenate((Kp_list, [Kp_list[-1]+np.diff(Kp_list)[-1]]))

    # Get spacings (deltas) between points
    delta_x = np.concatenate((np.diff(x_coords), [np.diff(x_coords)[-1]]))
    delta_y = np.concatenate((np.diff(y_coords), [np.diff(y_coords)[-1]]))

    # Center coordinates by half a delta
    x_coords = x_coords - 0.5 * delta_x
    y_coords = y_coords - 0.5 * delta_y

    # Create meshgrid for plotting
    XX_k, YY_k = np.meshgrid(x_coords, y_coords)

    ######################################
    # 2. Initialize Kp vs. Vsys map and variables
    ######################################
    RV_min = 1e6
    RV_max = 1e-6

    phases = OBS_OBJECT.phase           # The phase of observations
    velocities = CCF_OBJECT.rv_grid     # The radial velocity grid
    CCF = CCF_OBJECT.map_prf            # The cross-correlation function array

    KpVsys_map = np.zeros((len(Kp_list), len(vsys_list)))  # Output map

    phi_min = -0.02
    phi_max = 0.02
    mask = ((phases > phi_min) & (phases < phi_max))
    idx = np.where(mask == True)[0][0]

    selected_phases = phases[mask]
    
    ######################################
    # 3. Fill Kp vs. Vsys map
    ######################################
    for i, Kp in enumerate(Kp_list):
        for j, vsys in enumerate(vsys_list):
            CCF_sum = 0
            for k, norm_phase in enumerate(selected_phases):
                # Compute the planet RV shift
                RV = vsys + Kp * np.sin(2*np.pi*norm_phase)

                # Sum the CCF for this phase and shift
                CCF_sum = CCF_sum + np.interp(RV, velocities, CCF[idx + k, :], left=0., right=0.)

                # Update min and max RV
                if RV.max() > RV_max:
                    RV_max = RV.max()
                if RV.min() < RV_min:
                    RV_min = RV.min()

            # Store total summed CCF for this (Kp, vsys) point
            KpVsys_map[i, j] = CCF_sum

    print('min RV =', RV_min, ' km/s | max RV = ', RV_max, ' km/s')

    ######################################
    # 4. Plotting the results
    ######################################
    fig = plt.figure(figsize=(10, 7))
    gs = gridspec.GridSpec(100, 42)

    ax1 = fig.add_subplot(gs[:96, 2:37])  # Main map
    axc = fig.add_subplot(gs[:96, 38:40]) # Colorbar

    # Choose normalization method
    if method == 'box':
        sigma = np.std(KpVsys_map[-50:, -50:])
        median = np.median(KpVsys_map)
        KpVsys_plot = (KpVsys_map - median) / sigma
        plot_title = 'BOX METHOD'
    else:
        # Sigma-clipping method
        masked_map = stats.sigma_clip(KpVsys_map, sigma=3, maxiters=4)
        sigma_clip = np.std(masked_map)
        med_clip = np.median(KpVsys_map)
        KpVsys_plot = (KpVsys_map - med_clip) / sigma_clip
        plot_title = 'SIGMA-CLIPPING METHOD'

    # Main map
    ax1.pcolormesh(XX_k, YY_k, KpVsys_plot, cmap='inferno')
    max_idx = np.unravel_index(KpVsys_map.argmax(), KpVsys_map.shape)
    x_max = vsys_list[max_idx[1]]
    y_max = Kp_list[max_idx[0]]

    # Mark the central (0,0) position and best-fit point
    ax1.axvline(0, color='w', linewidth=2, linestyle='--')
    ax1.axhline(0, color='w', linewidth=2, linestyle='--')
    ax1.plot([x_max], [y_max], 'ko', zorder=10)

    # Labels and title
    ax1.set_title(plot_title, fontsize=25)
    ax1.set_xlabel('$\Delta$ Vsys (km/s)', fontsize=25)
    ax1.set_ylabel('$\Delta$ Kp (km/s)', fontsize=25)
    ax1.tick_params(labelsize=20)

    # Colorbar
    cmap = matplotlib.cm.inferno
    norm = matplotlib.colors.Normalize(vmin=np.min(KpVsys_plot), vmax=np.max(KpVsys_plot))
    cb = matplotlib.colorbar.ColorbarBase(axc, cmap=cmap, norm=norm, orientation='vertical')
    cb.set_label(label='SNR', fontsize=25)
    cb.ax.tick_params(labelsize=20)

    plt.tight_layout()
    plt.show()


In [ ]:
#call the above function to create two other normalization methodes

Kp_list = np.linspace(-80,80,161)
vsys_list = np.linspace(-25,25,101)

calculate_KpVsys_map(obs, ccf_obj, Kp_list, vsys_list, method = 'box')
calculate_KpVsys_map(obs, ccf_obj, Kp_list, vsys_list, method = 'clip')